In [39]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import os
import nested_pandas as npd
import pyarrow as pa
from progressbar import progressbar
import pyarrow.parquet as pq
import pyarrow.compute as pc

## Separate only fields in common with feature analysis

In [2]:
used_fields = ['436', '512', '594', '641', '686', '705', '727', '770', '778', '805']

In [5]:
dirname = ['/media/snad/ztf_features/dr24-features_astra-clr_pretrained/embed_no_lc/' + \
          'ztfdr24_astra_embeddings/dataset/Norder=' +  str(i) +'/' for i in range(4, 8) ]

In [7]:
k = 3
flist = os.listdir(dirname[k])

In [8]:
flist

['Dir=50000', 'Dir=60000', 'Dir=110000', 'Dir=120000']

In [9]:
data_list = []

for fname in progressbar(flist):
    data_temp = pd.read_parquet(dirname[k] + '/' + fname)
    mask = data_temp['koid'].apply(lambda x: str(x)[:3] in used_fields)

    if sum(mask) > 0:
        data_list.append(data_temp[mask])

        print(sum(mask))

 25% (1 of 4) |######                    | Elapsed Time: 0:01:59 ETA:   0:05:57

438059


100% (4 of 4) |##########################| Elapsed Time: 0:03:44 Time:  0:03:440:47


42283


In [10]:
data_field = pd.concat(data_list, ignore_index=True)

In [11]:
data_field.to_parquet('data/fields_Norder7.parquet')

## Join all the common objects in the same file

In [3]:
flist = os.listdir('data/')

In [41]:
data_list = []
for fname in flist:
    table = pq.read_table("data/" + fname)
    
    target_type = pa.list_(pa.float32())
    for col in ["embedding_beggining", "embedding_middle", "embedding_end"]:
        table = table.set_column(table.schema.get_field_index(col), col,
                                  pc.cast(table[col], target_type))
    data_temp = table.to_pandas()

    data_list.append(data_temp)
data_fields = pd.concat(data_list, ignore_index=True)

In [42]:
data_fields.shape

(878132, 6)

### Average over embeddings

In [43]:
def embed_to_numpy(df, col_name, NDIM=512):
    pa_array = pa.array(df[col_name])
    np_1d_array = pa_array.values.to_numpy(zero_copy_only=False)
    np_2d_array = np_1d_array.reshape(-1, NDIM)
    return np_2d_array

In [44]:
emb_beg = embed_to_numpy(data_fields, "embedding_beggining")
emb_mid = embed_to_numpy(data_fields, "embedding_middle")
emb_end = embed_to_numpy(data_fields, "embedding_end")
emb_avg = 1./3. * (emb_beg + emb_mid + emb_end)


In [49]:
data_fields['embedding_average'] = list(emb_avg)

## Use only extragalactic fields

In [63]:
galactic_fields = ['436', '512', '594', '641', '686', '705', '727', '770', '778', '805']
extrag_fields = ['252', '401', '468', '626', '673', '718', '759', '795', '797', '848']

In [68]:
mask = data_fields['koid'].apply(lambda x: str(x)[:3] in extrag_fields)

In [69]:
data_use = data_fields[mask]

In [70]:
data_use.shape

(0, 7)